In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from collections import Counter
import ast
import re
import matplotlib.gridspec as gridspec


from matplotlib.colors import LinearSegmentedColormap
from matplotlib import colors as mcolors
from scipy.stats import linregress
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.cluster import KMeans
from tabulate import tabulate
from collections import Counter

%matplotlib inline


In [ ]:
data_customer=pd.read_excel('./../data/customer_data_all_process_v3.xlsx').drop(columns=['Unnamed: 0'])
data_customer

In [ ]:
data_customer_updated=data_customer.drop(columns=['sum_pendapatan_idr','min_pendapatan_idr',
       'max_pendapatan_idr','median_pendapatan_idr','sum_product_Pendapatan (Beban) Non Usaha','sum_product_Pendapatan Pengusahaan Properti, Air & Listrik',
       'sum_product_Pendapatan Pelayanan Kapal & Lainnya','sum_product_Pendapatan Rupa-Rupa Usaha',
       'sum_product_Pendapatan (Beban) Keuangan','sum_product_Pendapatan Barang Non Petikemas','sum_product_Pelayanan Petikemas',
       ])

In [ ]:
data_customer_updated.columns

In [ ]:
sns.set_style('whitegrid')
corr = data_customer_updated.drop(columns=['Cust No.']).corr()
colors = ['#ff6200', '#ffcaa8', 'white', '#ffcaa8', '#ff6200']
my_cmap = LinearSegmentedColormap.from_list('custom_map', colors, N=256)

In [ ]:
mask = np.zeros_like(corr)
mask[np.triu_indices_from(mask, k=1)] = True
plt.figure(figsize=(14, 12))
sns.heatmap(corr, mask=mask, cmap=my_cmap, annot=True, center=0, fmt='.2f', linewidths=2,annot_kws={'size': 8})
plt.title('Correlation Matrix', fontsize=10)
plt.show()

In [ ]:
scaler = StandardScaler()
columns_to_exclude = ['Cust No.']
columns_to_scale = data_customer_updated.columns.difference(columns_to_exclude)
customer_data_scaled = data_customer_updated.copy()
customer_data_scaled[columns_to_scale] = scaler.fit_transform(customer_data_scaled[columns_to_scale])
customer_data_scaled.head()

In [ ]:
customer_data_scaled.columns

# Dimension Reduction

1. Multicollinearity Detected: Based on the previous, we identified that our dataset contains multicollinear features. Dimensionality reduction can help us remove redundant information and alleviate the multicollinearity issue.
2. Better Clustering with K-means: Since K-means is a distance-based algorithm, having a large number of features can sometimes dilute the meaningful underlying patterns in the data. By reducing the dimensionality, we can help K-means to find more compact and well-separated clusters.
3. Noise Reduction: By focusing only on the most important features, we can potentially remove noise in the data, leading to more accurate and stable clusters.
4. Enhanced Visualization: In the context of customer segmentation, being able to visualize customer groups in two or three dimensions can provide intuitive insights. Dimensionality reduction techniques can facilitate this by reducing the data to a few principal components which can be plotted easily.
5. Improved Computational Efficiency: Reducing the number of features can speed up the computation time during the modeling process, making our clustering algorithm more efficient.

# Objectives:
To simplify our data while retaining the essential information. Among various methods such as KernelPCA, ICA, ISOMAP, TSNE, and UMAP, On this POC,starting with PCA (Principal Component Analysis). Here's why:

1. PCA is an excellent starting point because it works well in capturing linear relationships in the data, which is particularly relevant given the multicollinearity we identified in our dataset.


In [ ]:
customer_data_scaled.set_index('Cust No.', inplace=True)

In [ ]:
pca = PCA().fit(customer_data_scaled)
explained_variance_ratio = pca.explained_variance_ratio_
cumulative_explained_variance = np.cumsum(explained_variance_ratio)
optimal_k = 12
sns.set(rc={'axes.facecolor': '#fcf0dc'}, style='darkgrid')
plt.figure(figsize=(20, 10))
barplot = sns.barplot(x=list(range(1, len(cumulative_explained_variance) + 1)),
                      y=explained_variance_ratio,
                      color='#fcc36d',
                      alpha=0.8)

lineplot, = plt.plot(range(0, len(cumulative_explained_variance)), cumulative_explained_variance,
                     marker='o', linestyle='--', color='#ff6200', linewidth=2)

optimal_k_line = plt.axvline(optimal_k - 1, color='red', linestyle='--', label=f'Optimal k value = {optimal_k}') 

plt.xlabel('Number of Components', fontsize=14)
plt.ylabel('Explained Variance', fontsize=14)
plt.title('Cumulative Variance vs. Number of Components', fontsize=18)

plt.xticks(range(0, len(cumulative_explained_variance)))
plt.legend(handles=[barplot.patches[0], lineplot, optimal_k_line],
           labels=['Explained Variance of Each Component', 'Cumulative Explained Variance', f'Optimal k value = {optimal_k}'],
           loc=(0.62, 0.1),
           frameon=True,
           framealpha=1.0,  
           edgecolor='#ff6200')  

x_offset = -0.3
y_offset = 0.01
for i, (ev_ratio, cum_ev_ratio) in enumerate(zip(explained_variance_ratio, cumulative_explained_variance)):
    plt.text(i, ev_ratio, f"{ev_ratio:.2f}", ha="center", va="bottom", fontsize=10)
    if i > 0:
        plt.text(i + x_offset, cum_ev_ratio + y_offset, f"{cum_ev_ratio:.2f}", ha="center", va="bottom", fontsize=10)

plt.grid(axis='both')   
plt.show()


# Insight:
1. The first component explains approximately 15% of the variance.
2. The first two components together explain about 26% of the variance.
3. The first three components explain approximately 33% of the variance, and so on.
4. Based on the plot, we can see the optimal component is 10

In [ ]:
pca = PCA(n_components=12)
customer_data_pca = pca.fit_transform(customer_data_scaled)
customer_data_pca = pd.DataFrame(customer_data_pca, columns=['PC'+str(i+1) for i in range(pca.n_components_)])
customer_data_pca.index = customer_data_scaled.index

In [ ]:
def highlight_top3(column):
    top3 = column.abs().nlargest(3).index
    return ['background-color:  #ffeacc' if i in top3 else '' for i in column.index]

In [ ]:
pc_df = pd.DataFrame(pca.components_.T, columns=['PC{}'.format(i+1) for i in range(pca.n_components_)],  
                     index=customer_data_scaled.columns)
pc_df.style.apply(highlight_top3, axis=0)

# Clustering

In [ ]:
sns.set(style='darkgrid', rc={'axes.facecolor': '#fcf0dc'})

sns.set_palette(['#ff6200'])
km = KMeans(init='k-means++', n_init=10, max_iter=100, random_state=0)
fig, ax = plt.subplots(figsize=(12, 5))
visualizer = KElbowVisualizer(km, k=(2, 15), timings=False, ax=ax)
visualizer.fit(customer_data_pca)
visualizer.show()

In [ ]:
# Eval
def silhouette_analysis(df, start_k, stop_k, figsize=(15, 16)):
    """
    Perform Silhouette analysis for a range of k values and visualize the results.
    """

    plt.figure(figsize=figsize)
    grid = gridspec.GridSpec(stop_k - start_k + 1, 2)
    first_plot = plt.subplot(grid[0, :])
    sns.set_palette(['darkorange'])

    silhouette_scores = []

    for k in range(start_k, stop_k + 1):
        km = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=100, random_state=0)
        km.fit(df)
        labels = km.predict(df)
        score = silhouette_score(df, labels)
        silhouette_scores.append(score)

    best_k = start_k + silhouette_scores.index(max(silhouette_scores))

    plt.plot(range(start_k, stop_k + 1), silhouette_scores, marker='o')
    plt.xticks(range(start_k, stop_k + 1))
    plt.xlabel('Number of clusters (k)')
    plt.ylabel('Silhouette score')
    plt.title('Average Silhouette Score for Different k Values', fontsize=15)

    optimal_k_text = f'The k value with the highest Silhouette score is: {best_k}'
    plt.text(10, 0.23, optimal_k_text, fontsize=12, verticalalignment='bottom', 
             horizontalalignment='left', bbox=dict(facecolor='#fcc36d', edgecolor='#ff6200', boxstyle='round, pad=0.5'))
             

    colors = sns.color_palette("bright")

    for i in range(start_k, stop_k + 1):    
        km = KMeans(n_clusters=i, init='k-means++', n_init=10, max_iter=100, random_state=0)
        row_idx, col_idx = divmod(i - start_k, 2)

        ax = plt.subplot(grid[row_idx + 1, col_idx])

        visualizer = SilhouetteVisualizer(km, colors=colors, ax=ax)
        visualizer.fit(df)

        score = silhouette_score(df, km.labels_)
        ax.text(0.97, 0.02, f'Silhouette Score: {score:.2f}', fontsize=12, \
                ha='right', transform=ax.transAxes, color='red')

        ax.set_title(f'Silhouette Plot for {i} Clusters', fontsize=15)

    plt.tight_layout()
    plt.show()

In [ ]:
silhouette_analysis(customer_data_pca, 3, 12, figsize=(20, 50))


# Sil Score Analysis
1. Silhouette Score Width:
a. Wide Widths (closer to +1): Indicate that the data points in the cluster are well separated from points in other clusters, suggesting well-defined clusters.
b. Narrow Widths (closer to -1): Show that data points in the cluster are not distinctly separated from other clusters, indicating poorly defined clusters.

2. Average Silhouette Score:
a. High Average Width: A cluster with a high average silhouette score indicates well-separated clusters.
b. Low Average Width: A cluster with a low average silhouette score indicates poor separation between clusters.

3. Peaks in Average Silhouette Score:
Clear Peaks: A clear peak in the average silhouette score plot for a specific ( k ) value indicates this ( k ) might be optimal.
4. Minimize Fluctuations in Silhouette Plot Widths:
a. Uniform Widths: Seek silhouette plots with similar widths across clusters, suggesting a more balanced and optimal clustering.
b. Variable Widths: Avoid wide fluctuations in silhouette plot widths, indicating that clusters are not well-defined and may vary in compactness.
5. Optimal Cluster Selection:
a. Maximize the Overall Average Silhouette Score: Choose the ( k ) value that gives the highest average silhouette score across all clusters, indicating well-defined clusters.
b. Avoid Below-Average Silhouette Scores: Ensure most clusters have above-average silhouette scores to prevent suboptimal clustering structures.
6. Visual Inspection of Silhouette Plots:
a. Consistent Cluster Formation: Visually inspect the silhouette plots for each ( k ) value to evaluate the consistency and structure of the formed clusters.
b. Cluster Compactness: Look for more compact clusters, with data points having silhouette scores closer to +1, indicating better clustering.

In [ ]:
kmeans = KMeans(n_clusters=6, init='k-means++', n_init=10, max_iter=100, random_state=0)
kmeans.fit(customer_data_pca)
cluster_frequencies = Counter(kmeans.labels_)
label_mapping = {label: new_label for new_label, (label, _) in 
                 enumerate(cluster_frequencies.most_common())}
new_labels = np.array([label_mapping[label] for label in kmeans.labels_])
data_customer_updated['cluster'] = new_labels
customer_data_pca['cluster'] = new_labels

In [ ]:
customer_data_pca['cluster'].value_counts()

# Profiling Cluster

In [ ]:
cluster_percentage = (customer_data_pca['cluster'].value_counts(normalize=True) * 100).reset_index()
cluster_percentage.columns = ['Cluster', 'Percentage']
cluster_percentage.sort_values(by='Cluster', inplace=True)
plt.figure(figsize=(10, 4))
sns.barplot(x='Percentage', y='Cluster', data=cluster_percentage, orient='h', palette=colors)

for index, value in enumerate(cluster_percentage['Percentage']):
    plt.text(value+0.5, index, f'{value:.2f}%')

plt.title('Distribution of Customers Across Clusters', fontsize=14)
plt.xticks(ticks=np.arange(0, 50, 5))
plt.xlabel('Percentage (%)')
plt.show()

In [ ]:
num_observations = len(customer_data_pca)
X = customer_data_pca.drop('cluster', axis=1)
clusters = customer_data_pca['cluster']

sil_score = silhouette_score(X, clusters)
calinski_score = calinski_harabasz_score(X, clusters)
davies_score = davies_bouldin_score(X, clusters)

table_data = [
    ["Number of Observations", num_observations],
    ["Silhouette Score", sil_score]
]

print(tabulate(table_data, headers=["Metric", "Value"], tablefmt='pretty'))


In [ ]:
df_customer = data_customer_updated.set_index('Cust No.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_customer_standardized = scaler.fit_transform(df_customer.drop(columns=['cluster'], axis=1))
df_customer_standardized = pd.DataFrame(df_customer_standardized, columns=df_customer.columns[:-1], index=df_customer.index)
df_customer_standardized['cluster'] = df_customer['cluster']

cluster_centroids = df_customer_standardized.groupby('cluster').mean()

def create_radar_chart(ax, angles, data, color, cluster):
    ax.fill(angles, data, color=color, alpha=0.4)
    ax.plot(angles, data, color=color, linewidth=2, linestyle='solid')
    ax.set_title(f'Cluster {cluster}', size=16, color=color, y=1.1)

labels = np.array(cluster_centroids.columns)
num_vars = len(labels)

angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
labels = np.concatenate((labels, [labels[0]]))
angles += angles[:1]

colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k', 'orange']

rows, cols = 4, 2
fig, ax = plt.subplots(figsize=(16, 20), subplot_kw=dict(polar=True), nrows=rows, ncols=cols)

ax = ax.flatten()

for i, color in enumerate(colors[:len(cluster_centroids)]):
    data = cluster_centroids.loc[i].tolist()
    data += data[:1]
    create_radar_chart(ax[i], angles, data, color, i)

    ax[i].set_xticks(angles[:-1])
    ax[i].set_xticklabels(labels[:-1], fontsize=10)

    ax[i].grid(color='grey', linewidth=0.5)
    ax[i].set_yticklabels([])
plt.tight_layout()
plt.savefig('./../results/plot/cluster_results_profile_v3.png',dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
features = data_customer_updated.columns[1:-1]
clusters = data_customer_updated['cluster'].unique()
clusters.sort()

n_rows = len(features)
n_cols = len(clusters)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 3*n_rows))
for i, feature in enumerate(features):
    for j, cluster in enumerate(clusters):
        data = data_customer[data_customer['cluster'] == cluster][feature]
        axes[i, j].hist(data, bins=20, color=colors[j], edgecolor='w', alpha=0.7)
        axes[i, j].set_title(f'Cluster {cluster} - {feature}', fontsize=15)
        axes[i, j].set_xlabel('')
        axes[i, j].set_ylabel('')

plt.tight_layout()
plt.savefig('./../results/plot/cluster_results_profile_v4.png',dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
data_customer_updated[['cluster']].value_counts()

In [ ]:
customer_data_pca.to_excel('./../results/file/customer_data_pca_v2_updated_feature.xlsx')

In [ ]:
customer_data_scaled.to_excel('./../results/file/customer_data_scaled_v2_updated_feature.xlsx')

In [ ]:
df_customer_standardized.to_excel('./../results/file/customer_data_scaled_standardized_v2_updated_feature.xlsx')

In [ ]:
data_customer_updated.to_excel('./../results/file/data_customer_v2_updated_feature.xlsx')

In [ ]:
data_customer_updated.columns

In [ ]:
import plotly.express as px

# Membuat fungsi untuk menentukan LEVEL_1 berdasarkan kondisi freq_product
def determine_level(row):
    # Daftar kolom freq_product
    freq_product_columns = [
        'freq_product_Pendapatan (Beban) Non Usaha',
        'freq_product_Pendapatan Pengusahaan Properti, Air & Listrik',
        'freq_product_Pendapatan Pelayanan Kapal & Lainnya',
        'freq_product_Pendapatan Rupa-Rupa Usaha',
        'freq_product_Pendapatan (Beban) Keuangan',
        'freq_product_Pendapatan Barang Non Petikemas',
        'freq_product_Pelayanan Petikemas'
    ]
    
    # Memeriksa jika ada freq_product yang lebih besar dari 0
    for col in freq_product_columns:
        if row[col] > 0:
            return col  # Kembali dengan kolom yang memiliki freq lebih besar dari 0
    
    return 'No Activity'  # Jika tidak ada freq_product lebih besar dari 0

# Menambahkan kolom LEVEL_1 berdasarkan hasil fungsi
data_customer_updated['LEVEL_1'] = data_customer_updated.apply(determine_level, axis=1)

# Menghitung jumlah Cust No. per LEVEL_1 dan cluster
cluster_count = data_customer_updated.groupby(['LEVEL_1', 'cluster'])['Cust No.'].nunique().reset_index()

# Menghitung total Cust No. per cluster
total_cust_per_cluster = cluster_count.groupby('cluster')['Cust No.'].sum().reset_index()

# Menggabungkan total Cust No. dengan cluster_count untuk menghitung persentase
cluster_count = cluster_count.merge(total_cust_per_cluster, on='cluster', suffixes=('', '_total'))

# Menghitung persentase untuk tiap LEVEL_1 per cluster
cluster_count['Percentage'] = (cluster_count['Cust No.'] / cluster_count['Cust No._total']) * 100

# Plotting menggunakan Plotly untuk interaktif
fig = px.bar(
    cluster_count, 
    x='Percentage', 
    y='LEVEL_1', 
    color='cluster', 
    orientation='h', 
    labels={'Percentage': 'Percentage (%)', 'LEVEL_1': 'Pendapatan Type'},
    hover_data={'Cust No.': True, 'Percentage': True, 'cluster': True}  # Menampilkan data yang relevan saat hover
)

# Menambahkan judul plot
fig.update_layout(
    title="Distribution of Pendapatan Types Based on Cust No. and Cluster",
    barmode='stack',  # Menggunakan stacking untuk menampilkan data
    xaxis_title="Count Cust No.",
    yaxis_title="LEVEL_1",
)

# Menampilkan plot interaktif
fig.show()


In [ ]:
cluster_count

# Trash

In [ ]:
flattened_list = [item for sublist in data_customer['prof_center'] for item in sublist]
frequency = Counter(flattened_list)
most_common_elements = frequency.most_common()
top_5_elements = most_common_elements[:20]
top_5_elements

## Aggregate Monthly to 1 Row per Customer

In [ ]:
customer_monthly_data = data_customer.groupby('Cust No.').agg({
    'jumlah_transaksi_pembelian': 'sum',  
    'jumlah_transaksi_retur': 'sum',      
    # 'pembelian_sum_pendapatan_idr': 'sum',
    # 'retur_sum_pendapatan_idr': 'sum',    
    # 'pembelian_median_pendapatan_idr': 'median',
    # 'retur_median_pendapatan_idr': 'median',    
    # 'pembelian_max_pendapatan_idr': 'max',      
    # 'retur_max_pendapatan_idr': 'max',     
}).reset_index()

In [ ]:
customer_monthly_data.describe()

In [ ]:
customer_monthly_data

## Normalization

In [ ]:
scaler = StandardScaler()
features_scaled = scaler.fit_transform(customer_monthly_data.drop(columns=['Cust No.']))
features_scaled

In [ ]:
scaled_data = pd.DataFrame(features_scaled, columns=customer_monthly_data.columns[1:])
scaled_data['Cust No.'] = customer_monthly_data['Cust No.']

## Modelling K Means

In [ ]:
inertia = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(features_scaled)
    inertia.append(kmeans.inertia_)

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(range(1, 11), inertia, marker='o')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
customer_monthly_data['Cluster'] = kmeans.fit_predict(features_scaled)
customer_monthly_data[['Cust No.', 'Cluster']].head()

In [ ]:
customer_monthly_data[['Cluster']].value_counts()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x=customer_monthly_data['jumlah_transaksi_pembelian'], 
                y=customer_monthly_data['jumlah_transaksi_retur'], 
                hue=customer_monthly_data['Cluster'], 
                palette='viridis', s=100, alpha=0.7)
plt.title('Clustering Pelanggan Berdasarkan Pembelian dan Retur')
plt.xlabel('Jumlah Transaksi Pembelian')
plt.ylabel('Jumlah Transaksi Retur')
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()